<a href="https://colab.research.google.com/github/Forla03/Deep-Learning-Project---July-2026/blob/main/Deep_Learning_Project_July_Forlani_Francesco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# /content/chess_white_to_black_project.py
"""
White-to-Black chess move prediction project.

Task:
    Given a sequence of White SAN moves, predict the aligned sequence of Black SAN moves.

Model:
    White move ids -> Embedding -> Bidirectional GRU -> bottleneck -> Black move softmax.

Optional inference boost:
    A train-only opening-prefix book overrides early moves when a White prefix was frequent enough
    in the training set. Disable USE_OPENING_BOOK if the evaluation must be purely neural.
"""

from __future__ import annotations

import json
import os
import random
import subprocess
import sys
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras


@dataclass(frozen=True)
class Config:
    TRAIN_CSV_PATH: str = "/content/white_to_black_train.csv"
    TEST_CSV_PATH: str = "/content/white_to_black_test.csv"

    # Se nel notebook sorgente ci sono URL diretti, incollali qui.
    TRAIN_CSV_URL: str = ""
    TEST_CSV_URL: str = ""

    MAX_LEN: int = 60
    WHITE_VOCAB_SIZE: int = 20_000
    BLACK_VOCAB_SIZE: int = 20_000

    EMBEDDING_DIM: int = 80
    RNN_UNITS: int = 128
    BOTTLENECK_DIM: int = 144
    DROPOUT: float = 0.20
    SPATIAL_DROPOUT: float = 0.10

    BATCH_SIZE: int = 64
    PREDICTION_BATCH_SIZE: int = 32
    EPOCHS: int = 25
    VALIDATION_FRACTION: float = 0.10
    LEARNING_RATE: float = 1e-3

    OPENING_BOOK_MAX_DEPTH: int = 14
    OPENING_BOOK_MIN_COUNT: int = 3
    USE_OPENING_BOOK: bool = True

    SEED: int = 42
    ARTIFACT_DIR: str = "/content/chess_white_to_black_artifacts"
    BEST_WEIGHTS_NAME: str = "white_to_black_bigru_best.weights.h5"
    FINAL_WEIGHTS_NAME: str = "white_to_black_bigru_final.weights.h5"
    METADATA_NAME: str = "white_to_black_metadata.json"


CFG = Config()


def set_reproducibility(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def configure_runtime() -> None:
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        keras.mixed_precision.set_global_policy("mixed_float16")
        print(f"GPU detected: {gpus[0].name}. Mixed precision enabled.")
    else:
        print("No GPU detected. Training will work, but it will be slower.")


def download_if_needed(path: str, url: str) -> None:
    target = Path(path)

    if target.exists():
        return

    if not url:
        raise FileNotFoundError(
            f"File mancante: {path}\n"
            f"Caricalo in Colab oppure imposta il relativo URL in Config."
        )

    target.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {target.name}...")
    tf.keras.utils.get_file(
        fname=target.name,
        origin=url,
        cache_dir=str(target.parent),
        cache_subdir=".",
    )


def load_and_clean_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    required_columns = {"white", "black"}
    missing_columns = required_columns.difference(df.columns)

    if missing_columns:
        raise ValueError(f"{path} non contiene le colonne richieste: {sorted(missing_columns)}")

    df = df.loc[:, ["white", "black"]].dropna()
    df["white"] = df["white"].astype(str).str.strip()
    df["black"] = df["black"].astype(str).str.strip()
    df = df[(df["white"] != "") & (df["black"] != "")].copy()

    white_lengths = df["white"].str.split().str.len()
    black_lengths = df["black"].str.split().str.len()
    df = df[white_lengths == black_lengths].copy()
    df = df.reset_index(drop=True)

    if df.empty:
        raise ValueError(f"Nessuna riga valida dopo la pulizia di {path}.")

    return df


def load_data(cfg: Config) -> tuple[pd.DataFrame, pd.DataFrame]:
    download_if_needed(cfg.TRAIN_CSV_PATH, cfg.TRAIN_CSV_URL)
    download_if_needed(cfg.TEST_CSV_PATH, cfg.TEST_CSV_URL)

    train_df = load_and_clean_csv(cfg.TRAIN_CSV_PATH)
    test_df = load_and_clean_csv(cfg.TEST_CSV_PATH)

    print(f"Train rows: {len(train_df):,}")
    print(f"Test rows:  {len(test_df):,}")
    print(
        "Average train length:",
        round(float(train_df["white"].str.split().str.len().mean()), 2),
    )

    return train_df, test_df


def split_train_validation(
    train_df: pd.DataFrame,
    validation_fraction: float,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    shuffled = train_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    val_size = max(1, int(len(shuffled) * validation_fraction))

    val_df = shuffled.iloc[:val_size].reset_index(drop=True)
    fit_df = shuffled.iloc[val_size:].reset_index(drop=True)

    return fit_df, val_df


def create_vectorizer(
    max_tokens: int,
    max_len: int,
    name: str,
) -> keras.layers.TextVectorization:
    return keras.layers.TextVectorization(
        max_tokens=max_tokens,
        standardize=None,
        split="whitespace",
        output_mode="int",
        output_sequence_length=max_len,
        name=name,
    )


def adapt_vectorizers(
    train_df: pd.DataFrame,
    cfg: Config,
) -> tuple[keras.layers.TextVectorization, keras.layers.TextVectorization]:
    white_vectorizer = create_vectorizer(
        cfg.WHITE_VOCAB_SIZE,
        cfg.MAX_LEN,
        "white_vectorizer",
    )
    black_vectorizer = create_vectorizer(
        cfg.BLACK_VOCAB_SIZE,
        cfg.MAX_LEN,
        "black_vectorizer",
    )

    white_ds = tf.data.Dataset.from_tensor_slices(train_df["white"].to_numpy()).batch(1024)
    black_ds = tf.data.Dataset.from_tensor_slices(train_df["black"].to_numpy()).batch(1024)

    white_vectorizer.adapt(white_ds)
    black_vectorizer.adapt(black_ds)

    print(f"White vocabulary size: {len(white_vectorizer.get_vocabulary()):,}")
    print(f"Black vocabulary size: {len(black_vectorizer.get_vocabulary()):,}")

    return white_vectorizer, black_vectorizer


def vectorize_dataframe(
    df: pd.DataFrame,
    white_vectorizer: keras.layers.TextVectorization,
    black_vectorizer: keras.layers.TextVectorization,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = white_vectorizer(tf.constant(df["white"].to_numpy())).numpy().astype("int32")
    y = black_vectorizer(tf.constant(df["black"].to_numpy())).numpy().astype("int32")
    sample_weight = (y != 0).astype("float32")

    return x, y, sample_weight


def make_dataset(
    x: np.ndarray,
    y: np.ndarray,
    sample_weight: np.ndarray,
    batch_size: int,
    shuffle: bool,
    seed: int,
) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices((x, y, sample_weight))

    if shuffle:
        ds = ds.shuffle(
            buffer_size=min(len(x), 20_000),
            seed=seed,
            reshuffle_each_iteration=True,
        )

    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def build_opening_book(
    train_df: pd.DataFrame,
    max_depth: int,
    min_count: int,
) -> dict[str, dict[str, Any]]:
    counters: dict[str, Counter[str]] = defaultdict(Counter)

    for white_text, black_text in zip(train_df["white"], train_df["black"], strict=False):
        white_moves = white_text.split()[:max_depth]
        black_moves = black_text.split()[:max_depth]

        for move_index, black_move in enumerate(black_moves[: len(white_moves)]):
            prefix = " ".join(white_moves[: move_index + 1])
            counters[prefix][black_move] += 1

    opening_book: dict[str, dict[str, Any]] = {}

    for prefix, counter in counters.items():
        move, count = counter.most_common(1)[0]
        total = sum(counter.values())

        if count >= min_count:
            opening_book[prefix] = {
                "move": move,
                "count": int(count),
                "total": int(total),
                "confidence": float(count / total),
            }

    print(f"Opening book entries: {len(opening_book):,}")
    return opening_book


def build_model(
    white_vocab_size: int,
    black_vocab_size: int,
    cfg: Config,
) -> keras.Model:
    inputs = keras.Input(shape=(cfg.MAX_LEN,), dtype="int32", name="white_tokens")

    x = keras.layers.Embedding(
        input_dim=white_vocab_size,
        output_dim=cfg.EMBEDDING_DIM,
        mask_zero=True,
        name="white_embedding",
    )(inputs)
    x = keras.layers.SpatialDropout1D(
        cfg.SPATIAL_DROPOUT,
        name="embedding_dropout",
    )(x)
    x = keras.layers.Bidirectional(
        keras.layers.GRU(
            cfg.RNN_UNITS,
            return_sequences=True,
            dropout=cfg.DROPOUT,
            recurrent_dropout=0.0,
        ),
        name="context_bigru",
    )(x)
    x = keras.layers.Dense(
        cfg.BOTTLENECK_DIM,
        activation="gelu",
        name="move_bottleneck",
    )(x)
    x = keras.layers.Dropout(cfg.DROPOUT, name="bottleneck_dropout")(x)

    outputs = keras.layers.Dense(
        black_vocab_size,
        activation="softmax",
        dtype="float32",
        name="black_move",
    )(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="white_to_black_bigru")

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=cfg.LEARNING_RATE,
            clipnorm=1.0,
        ),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="token_acc")],
    )

    return model


def assert_parameter_budget(
    model: keras.Model,
    max_trainable_params: int = 6_000_000,
) -> None:
    trainable_params = int(
        np.sum([np.prod(weight.shape) for weight in model.trainable_weights])
    )
    print(f"Trainable parameters: {trainable_params:,}")

    if trainable_params >= max_trainable_params:
        raise ValueError(
            f"Il modello ha {trainable_params:,} parametri trainabili; "
            f"deve restare sotto {max_trainable_params:,}."
        )


def train_model(
    model: keras.Model,
    train_ds: tf.data.Dataset,
    val_ds: tf.data.Dataset,
    cfg: Config,
) -> keras.callbacks.History:
    artifact_dir = Path(cfg.ARTIFACT_DIR)
    artifact_dir.mkdir(parents=True, exist_ok=True)

    best_weights_path = artifact_dir / cfg.BEST_WEIGHTS_NAME

    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(best_weights_path),
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
            mode="min",
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            mode="min",
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
            mode="min",
            verbose=1,
        ),
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=cfg.EPOCHS,
        callbacks=callbacks,
    )

    if best_weights_path.exists():
        model.load_weights(str(best_weights_path))

    return history


def ids_to_tokens(
    ids: np.ndarray,
    vocabulary: list[str],
    limit: int,
) -> list[str]:
    decoded: list[str] = []

    for token_id in ids[:limit]:
        token_id_int = int(token_id)

        if token_id_int <= 0 or token_id_int >= len(vocabulary):
            break

        token = vocabulary[token_id_int]

        if token == "":
            break

        decoded.append(token)

    return decoded


def apply_opening_book(
    white_text: str,
    predicted_tokens: list[str],
    opening_book: dict[str, dict[str, Any]] | None,
    cfg: Config,
) -> list[str]:
    if not cfg.USE_OPENING_BOOK or not opening_book:
        return predicted_tokens

    white_moves = white_text.split()
    output = list(predicted_tokens)
    limit = min(len(output), len(white_moves), cfg.OPENING_BOOK_MAX_DEPTH)

    for index in range(limit):
        prefix = " ".join(white_moves[: index + 1])
        entry = opening_book.get(prefix)

        if entry and int(entry["count"]) >= cfg.OPENING_BOOK_MIN_COUNT:
            output[index] = str(entry["move"])

    return output


def predict_black_moves_batch(
    model: keras.Model,
    white_texts: list[str],
    white_vectorizer: keras.layers.TextVectorization,
    black_vocabulary: list[str],
    cfg: Config,
    opening_book: dict[str, dict[str, Any]] | None = None,
) -> list[list[str]]:
    x = white_vectorizer(tf.constant(white_texts)).numpy().astype("int32")
    probabilities = model.predict(
        x,
        batch_size=cfg.PREDICTION_BATCH_SIZE,
        verbose=0,
    )
    predicted_ids = np.argmax(probabilities, axis=-1)

    decoded_batch: list[list[str]] = []

    for white_text, row_ids in zip(white_texts, predicted_ids, strict=False):
        limit = min(len(white_text.split()), cfg.MAX_LEN)
        decoded = ids_to_tokens(row_ids, black_vocabulary, limit)
        decoded = apply_opening_book(white_text, decoded, opening_book, cfg)
        decoded_batch.append(decoded[:limit])

    return decoded_batch


def predict_black_moves(
    model: keras.Model,
    white_text: str,
    white_vectorizer: keras.layers.TextVectorization,
    black_vocabulary: list[str],
    cfg: Config,
    opening_book: dict[str, dict[str, Any]] | None = None,
) -> list[str]:
    return predict_black_moves_batch(
        model=model,
        white_texts=[white_text],
        white_vectorizer=white_vectorizer,
        black_vocabulary=black_vocabulary,
        cfg=cfg,
        opening_book=opening_book,
    )[0]


def correct_prefix_length(
    true_tokens: list[str],
    predicted_tokens: list[str],
) -> int:
    correct = 0

    for true_token, predicted_token in zip(true_tokens, predicted_tokens, strict=False):
        if true_token != predicted_token:
            break

        correct += 1

    return correct


def batched_ranges(length: int, batch_size: int) -> list[tuple[int, int]]:
    return [
        (start, min(start + batch_size, length))
        for start in range(0, length, batch_size)
    ]


def evaluate_survival_auc(
    dataframe: pd.DataFrame,
    model: keras.Model,
    white_vectorizer: keras.layers.TextVectorization,
    black_vocabulary: list[str],
    cfg: Config,
    opening_book: dict[str, dict[str, Any]] | None = None,
    max_examples: int | None = None,
) -> dict[str, Any]:
    eval_df = dataframe if max_examples is None else dataframe.head(max_examples)
    eval_df = eval_df.reset_index(drop=True)

    prefix_lengths: list[int] = []
    token_correct = 0
    token_total = 0
    first_move_correct = 0
    exact_sequence_correct = 0

    for start, end in batched_ranges(len(eval_df), cfg.PREDICTION_BATCH_SIZE):
        batch = eval_df.iloc[start:end]
        white_texts = batch["white"].tolist()
        true_black_texts = batch["black"].tolist()

        predictions = predict_black_moves_batch(
            model=model,
            white_texts=white_texts,
            white_vectorizer=white_vectorizer,
            black_vocabulary=black_vocabulary,
            cfg=cfg,
            opening_book=opening_book,
        )

        for true_text, predicted_tokens in zip(true_black_texts, predictions, strict=False):
            true_tokens = true_text.split()[: cfg.MAX_LEN]
            predicted_tokens = predicted_tokens[: len(true_tokens)]

            prefix_len = correct_prefix_length(true_tokens, predicted_tokens)
            prefix_lengths.append(prefix_len)

            first_move_correct += int(prefix_len >= 1)
            exact_sequence_correct += int(prefix_len == len(true_tokens))

            for true_token, predicted_token in zip(true_tokens, predicted_tokens, strict=False):
                token_correct += int(true_token == predicted_token)
                token_total += 1

    max_curve_len = min(
        cfg.MAX_LEN,
        max((len(text.split()) for text in eval_df["black"]), default=0),
    )
    survival_curve = {
        int(k): float(np.mean([prefix_len >= k for prefix_len in prefix_lengths]))
        for k in range(1, max_curve_len + 1)
    }

    survival_auc = float(np.mean(prefix_lengths)) if prefix_lengths else 0.0

    return {
        "n_examples": int(len(eval_df)),
        "survival_auc": survival_auc,
        "mean_correct_prefix": survival_auc,
        "first_move_accuracy": (
            float(first_move_correct / len(eval_df)) if len(eval_df) else 0.0
        ),
        "exact_sequence_accuracy": (
            float(exact_sequence_correct / len(eval_df)) if len(eval_df) else 0.0
        ),
        "token_accuracy": float(token_correct / token_total) if token_total else 0.0,
        "survival_curve": survival_curve,
        "prefix_lengths": prefix_lengths,
    }


def print_metrics(metrics: dict[str, Any]) -> None:
    print(f"Examples:                {metrics['n_examples']:,}")
    print(f"Survival AUC:            {metrics['survival_auc']:.4f}")
    print(f"First move accuracy:     {metrics['first_move_accuracy']:.4f}")
    print(f"Token accuracy:          {metrics['token_accuracy']:.4f}")
    print(f"Exact sequence accuracy: {metrics['exact_sequence_accuracy']:.4f}")


def plot_survival_curve(metrics: dict[str, Any]) -> None:
    import matplotlib.pyplot as plt

    curve = metrics["survival_curve"]

    if not curve:
        print("No survival curve available.")
        return

    xs = list(curve.keys())
    ys = list(curve.values())

    plt.figure(figsize=(8, 5))
    plt.plot(xs, ys, marker="o")
    plt.xlabel("k")
    plt.ylabel("S(k)")
    plt.title("Survival Curve")
    plt.grid(True)
    plt.show()


def save_artifacts(
    model: keras.Model,
    white_vectorizer: keras.layers.TextVectorization,
    black_vectorizer: keras.layers.TextVectorization,
    opening_book: dict[str, dict[str, Any]],
    cfg: Config,
) -> tuple[Path, Path]:
    artifact_dir = Path(cfg.ARTIFACT_DIR)
    artifact_dir.mkdir(parents=True, exist_ok=True)

    final_weights_path = artifact_dir / cfg.FINAL_WEIGHTS_NAME
    metadata_path = artifact_dir / cfg.METADATA_NAME

    model.save_weights(str(final_weights_path))

    metadata = {
        "config": asdict(cfg),
        "white_vocabulary": white_vectorizer.get_vocabulary(),
        "black_vocabulary": black_vectorizer.get_vocabulary(),
        "opening_book": opening_book,
    }
    metadata_path.write_text(
        json.dumps(metadata, ensure_ascii=False),
        encoding="utf-8",
    )

    print(f"Saved weights:  {final_weights_path}")
    print(f"Saved metadata: {metadata_path}")

    return final_weights_path, metadata_path


def load_vectorizer_from_vocabulary(
    vocabulary: list[str],
    max_tokens: int,
    max_len: int,
    name: str,
) -> keras.layers.TextVectorization:
    vectorizer = create_vectorizer(
        max_tokens=max_tokens,
        max_len=max_len,
        name=name,
    )
    vectorizer.set_vocabulary(vocabulary)
    return vectorizer


def load_artifacts(
    artifact_dir: str,
    weights_name: str | None = None,
    metadata_name: str | None = None,
) -> tuple[
    keras.Model,
    keras.layers.TextVectorization,
    keras.layers.TextVectorization,
    dict[str, dict[str, Any]],
    Config,
]:
    artifact_path = Path(artifact_dir)
    metadata_path = artifact_path / (metadata_name or CFG.METADATA_NAME)

    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    loaded_cfg = Config(**metadata["config"])

    if weights_name is not None:
        loaded_cfg = Config(
            **{
                **asdict(loaded_cfg),
                "FINAL_WEIGHTS_NAME": weights_name,
            }
        )

    white_vectorizer = load_vectorizer_from_vocabulary(
        metadata["white_vocabulary"],
        loaded_cfg.WHITE_VOCAB_SIZE,
        loaded_cfg.MAX_LEN,
        "white_vectorizer",
    )
    black_vectorizer = load_vectorizer_from_vocabulary(
        metadata["black_vocabulary"],
        loaded_cfg.BLACK_VOCAB_SIZE,
        loaded_cfg.MAX_LEN,
        "black_vectorizer",
    )

    model = build_model(
        white_vocab_size=len(metadata["white_vocabulary"]),
        black_vocab_size=len(metadata["black_vocabulary"]),
        cfg=loaded_cfg,
    )
    model.load_weights(str(artifact_path / loaded_cfg.FINAL_WEIGHTS_NAME))

    opening_book = metadata.get("opening_book", {})

    return model, white_vectorizer, black_vectorizer, opening_book, loaded_cfg


def install_gdown() -> None:
    try:
        import gdown  # noqa: F401
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "gdown"],
            check=True,
        )


def download_weights_with_gdown(file_id: str, output_path: str) -> None:
    install_gdown()
    subprocess.run(
        ["gdown", "--id", file_id, "-O", output_path],
        check=True,
    )


def verify_gdown_weights(
    file_id: str,
    local_output_path: str,
    metadata_dir: str,
) -> None:
    download_weights_with_gdown(file_id, local_output_path)

    model, _, _, _, loaded_cfg = load_artifacts(
        artifact_dir=metadata_dir,
        weights_name=Path(local_output_path).name,
    )

    assert_parameter_budget(model)
    print(f"gdown weights loaded correctly from: {local_output_path}")
    print(f"Loaded config MAX_LEN: {loaded_cfg.MAX_LEN}")


def main() -> None:
    set_reproducibility(CFG.SEED)
    configure_runtime()

    train_df, test_df = load_data(CFG)

    fit_df, val_df = split_train_validation(
        train_df,
        validation_fraction=CFG.VALIDATION_FRACTION,
        seed=CFG.SEED,
    )

    white_vectorizer, black_vectorizer = adapt_vectorizers(train_df, CFG)

    x_train, y_train, sw_train = vectorize_dataframe(
        fit_df,
        white_vectorizer,
        black_vectorizer,
    )
    x_val, y_val, sw_val = vectorize_dataframe(
        val_df,
        white_vectorizer,
        black_vectorizer,
    )

    train_ds = make_dataset(
        x_train,
        y_train,
        sw_train,
        batch_size=CFG.BATCH_SIZE,
        shuffle=True,
        seed=CFG.SEED,
    )
    val_ds = make_dataset(
        x_val,
        y_val,
        sw_val,
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        seed=CFG.SEED,
    )

    validation_opening_book = build_opening_book(
        fit_df,
        max_depth=CFG.OPENING_BOOK_MAX_DEPTH,
        min_count=CFG.OPENING_BOOK_MIN_COUNT,
    )
    final_opening_book = build_opening_book(
        train_df,
        max_depth=CFG.OPENING_BOOK_MAX_DEPTH,
        min_count=CFG.OPENING_BOOK_MIN_COUNT,
    )

    model = build_model(
        white_vocab_size=len(white_vectorizer.get_vocabulary()),
        black_vocab_size=len(black_vectorizer.get_vocabulary()),
        cfg=CFG,
    )

    model.summary()
    assert_parameter_budget(model)

    train_model(model, train_ds, val_ds, CFG)

    black_vocabulary = black_vectorizer.get_vocabulary()

    print("\nValidation metrics")
    val_metrics = evaluate_survival_auc(
        dataframe=val_df,
        model=model,
        white_vectorizer=white_vectorizer,
        black_vocabulary=black_vocabulary,
        cfg=CFG,
        opening_book=validation_opening_book,
        max_examples=None,
    )
    print_metrics(val_metrics)
    plot_survival_curve(val_metrics)

    print("\nFull test metrics")
    test_metrics = evaluate_survival_auc(
        dataframe=test_df,
        model=model,
        white_vectorizer=white_vectorizer,
        black_vocabulary=black_vocabulary,
        cfg=CFG,
        opening_book=final_opening_book,
        max_examples=None,
    )
    print_metrics(test_metrics)
    plot_survival_curve(test_metrics)

    save_artifacts(
        model=model,
        white_vectorizer=white_vectorizer,
        black_vectorizer=black_vectorizer,
        opening_book=final_opening_book,
        cfg=CFG,
    )

    demo_white = test_df.iloc[0]["white"]
    demo_true_black = test_df.iloc[0]["black"]
    demo_pred_black = predict_black_moves(
        model=model,
        white_text=demo_white,
        white_vectorizer=white_vectorizer,
        black_vocabulary=black_vocabulary,
        cfg=CFG,
        opening_book=final_opening_book,
    )

    print("\nDemo")
    print("White:      ", demo_white)
    print("True black: ", demo_true_black)
    print("Pred black: ", " ".join(demo_pred_black))

    print("\nPer la consegna:")
    print(f"1. Carica su Google Drive: {Path(CFG.ARTIFACT_DIR) / CFG.FINAL_WEIGHTS_NAME}")
    print("2. Imposta condivisione: Anyone with the link.")
    print("3. Copia il file_id del link Drive.")
    print("4. Verifica con:")
    print(
        "verify_gdown_weights("
        "file_id='INCOLLA_FILE_ID', "
        f"local_output_path='{CFG.ARTIFACT_DIR}/{CFG.FINAL_WEIGHTS_NAME}', "
        f"metadata_dir='{CFG.ARTIFACT_DIR}'"
        ")"
    )


if __name__ == "__main__":
    main()